In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/04.Gold_Helper

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
results_table    = f"{catalog_name}.{silver_schema}.results"
sprint_table = f"{catalog_name}.{silver_schema}.sprints"

target_table = f"{catalog_name}.{gold_schema}.results_fact"

In [0]:
from pyspark.sql.functions import *

In [0]:
result_df = spark.read.table(results_table).filter(col("batch_id")==v_batch_id)

result_df = (
    result_df.withColumn('session_type', lit('RACE'))
             .drop('race_date','race_name','ingestion_timestamp','source_files','batch_id','created_timestamp','updated_timestamp')
)


In [0]:
sprints_df = spark.read.table(sprint_table).filter(col("batch_id")==v_batch_id)
sprints_df = (
    sprints_df.withColumn('session_type', lit('SPRINT'))
             .drop('race_date','race_name','ingestion_timestamp','source_files','batch_id','created_timestamp','updated_timestamp')
) 

In [0]:
sprint_result_df=(
    sprints_df
    .unionByName(result_df)
)

In [0]:
sprint_result_fact_df=(
    sprint_result_df
    .withColumn('is_win',col("finsh_position")==1)
    .withColumn('is_podium',col("finsh_position").between(1,3))
    .withColumn('has_points', col("points")>0)
)

In [0]:
 columns_to_update=[
        "grid_position",
        "completed_laps",
        "car_number",
        "points",
        "finsh_position",
        "finish_position_text",
        "status",
        "is_win",
        "is_podium",
        "has_points"
    ]

In [0]:
write_to_gold(input_df=sprint_result_fact_df,target_table=target_table,
                merge_condn="""
                                t.season = s.season
                                AND t.round = s.round
                                AND t.constructor_id = s.constructor_id
                                AND t.driver_id = s.driver_id
                                AND t.session_type = s.session_type
                            """,
                columns_to_update=columns_to_update
                )

In [0]:
display(spark.table(target_table))